In [23]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM


from tensorflow.keras.layers import (
    Conv1D,
    Dense,
    Dropout,
    GlobalMaxPooling1D
)

from tensorflow.keras.optimizers import Adam
from scipy.stats import t


In [16]:
DATASET_PATH = r"/kaggle/input/datasets/abubakarsiddiquemahi/phishing-url-dataset-11k-mendely/dataset_phishing(11k) Mendely.csv"

BATCH_SIZE = 16

LEARNING_RATE = 0.001

EPOCHS = 30

SEEDS = [42,3,7,72,82]


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    tf.random.set_seed(seed)


# ------------------------------------------------------------
# LOAD DATASET
# ------------------------------------------------------------

data = pd.read_csv(DATASET_PATH)

print(data.shape)

print(data.head())

print(data.columns)


# ------------------------------------------------------------
# HANDLE MISSING VALUES
# ------------------------------------------------------------

numeric_cols = data.select_dtypes(include=['number']).columns

categorical_cols = data.select_dtypes(exclude=['number']).columns


data[numeric_cols] = data[numeric_cols].fillna(
    data[numeric_cols].mean()
)


data[categorical_cols] = data[categorical_cols].fillna(
    "unknown"
)


# ------------------------------------------------------------
# LABEL ENCODING
# ------------------------------------------------------------

encoder = LabelEncoder()

for col in categorical_cols:

    data[col] = encoder.fit_transform(data[col])


# ------------------------------------------------------------
# FEATURES AND LABEL
# ------------------------------------------------------------

X = data.drop("status", axis=1)

y = data["status"]


# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(X)


# ------------------------------------------------------------
# TRAIN TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)


# ------------------------------------------------------------
# RESHAPE FOR 1D-CNN
# ------------------------------------------------------------

X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)


X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)


# ------------------------------------------------------------
# DISPLAY SHAPE
# ------------------------------------------------------------

print()

print("Training samples :", len(X_train))

print("Testing samples  :", len(X_test))

print()

print("Training shape :", X_train.shape)

print("Testing shape  :", X_test.shape)

(11430, 89)
                                                 url  length_url  \
0              http://www.crestonwood.com/router.php          37   
1  http://shadetreetechnology.com/V4/validation/a...          77   
2  https://support-appleld.com.secureupdate.duila...         126   
3                                 http://rgipt.ac.in          18   
4  http://www.iracing.com/tracks/gateway-motorspo...          55   

   length_hostname  ip  nb_dots  nb_hyphens  nb_at  nb_qm  nb_and  nb_or  ...  \
0               19   0        3           0      0      0       0      0  ...   
1               23   1        1           0      0      0       0      0  ...   
2               50   1        4           1      0      1       2      0  ...   
3               11   0        2           0      0      0       0      0  ...   
4               15   0        2           2      0      0       0      0  ...   

   domain_in_title  domain_with_copyright  whois_registered_domain  \
0                0    

# CNN

In [25]:



def create_model(input_shape):

    model = Sequential()

    model.add(
        Input(shape=input_shape)
    )

    model.add(
        Conv1D(
            filters=64,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Conv1D(
            filters=32,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        GlobalMaxPooling1D()
    )

    model.add(
        Dense(
            32,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

In [27]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (1D-CNN)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # Build 1D-CNN Model
    model = create_model(
        (X_train.shape[1], X_train.shape[2])
    )

    # Train Model
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )

    # =====================================================
    # Test Loss
    # =====================================================

    evaluation = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    test_loss = evaluation[0]

    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.6175 - auc: 0.6548 - loss: 0.6658 - val_accuracy: 0.6527 - val_auc: 0.7173 - val_loss: 0.6432
Epoch 2/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6459 - auc: 0.6929 - loss: 0.6350 - val_accuracy: 0.6745 - val_auc: 0.7556 - val_loss: 0.6239
Epoch 3/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6829 - auc: 0.7482 - loss: 0.5987 - val_accuracy: 0.7983 - val_auc: 0.8963 - val_loss: 0.4967
Epoch 4/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7695 - auc: 0.8442 - loss: 0.4939 - val_accuracy: 0.8513 - val_auc: 0.9289 - val_loss: 0.3650
Epoch 5/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8146 - auc: 0.8900 - loss: 0.4216 - val_accuracy: 0.8635 - val_auc: 0.9361 - val_loss: 0.3263
Epoch 6/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8351 - auc: 0.9092 - loss: 0.3864 - val_accuracy: 0.8753 - val_auc: 0.9434 - val_loss: 0.3092
Epoch 7/30
572/572 ━

In [28]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.222411  0.914698   0.943820  0.881890  0.911805  0.831188   
1     3  0.225190  0.908574   0.913274  0.902887  0.908051  0.817201   
2     7  0.218963  0.911636   0.927339  0.893263  0.909982  0.823828   
3    72  0.268734  0.895451   0.961224  0.824147  0.887423  0.799068   
4    82  0.232133  0.905512   0.943541  0.862642  0.901280  0.814021   

        AUC  Specificity  
0  0.970865     0.947507  
1  0.968717     0.914261  
2  0.970462     0.930009  
3  0.968633     0.966754  
4  0.971364     0.948381  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.233486  0.020289  0.2335 ± 0.0203  [0.2083, 0.2587]
1     Accuracy  0.907174  0.007394  0.9072 ± 0.0074  [0.8980, 0.9164]
2    Precision  0.937840  0.018227  0.9378 ± 0.0182  [0.9152, 0.9605]
3       Recall  0.872966  0.031132  0.8730 ± 0.0311  [0.8343, 0.9116]
4

# LSTM

In [31]:



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        LSTM(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        LSTM(
            64
        )
    )

    model.add(
        Dropout(0.5
        )
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [32]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (LSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build LSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.7150 - auc: 0.7731 - loss: 0.5748 - val_accuracy: 0.7130 - val_auc: 0.7720 - val_loss: 0.5844
Epoch 2/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.7626 - auc: 0.8303 - loss: 0.5095 - val_accuracy: 0.7922 - val_auc: 0.8685 - val_loss: 0.4687
Epoch 3/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8465 - auc: 0.9195 - loss: 0.3624 - val_accuracy: 0.9042 - val_auc: 0.9608 - val_loss: 0.2541
Epoch 4/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9075 - auc: 0.9621 - loss: 0.2447 - val_accuracy: 0.9103 - val_auc: 0.9656 - val_loss: 0.2339
Epoch 5/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9183 - auc: 0.9686 - loss: 0.2211 - val_accuracy: 0.9090 - val_auc: 0.9700 - val_loss: 0.2271
Epoch 6/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9231 - auc: 0.9703 - loss: 0.2140 - val_accuracy: 0.9239 - val_auc: 0.9727 - val_loss: 0.2100
Epoch 7/30
572/

In [33]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.161947  0.948819   0.952381  0.944882  0.948617  0.897666   
1     3  0.170166  0.943132   0.942358  0.944007  0.943182  0.886266   
2     7  0.170050  0.944444   0.953571  0.934383  0.943880  0.889069   
3    72  0.176990  0.940507   0.951570  0.928259  0.939770  0.881279   
4    82  0.193811  0.928696   0.946266  0.909011  0.927265  0.858058   

        AUC  Specificity  
0  0.983977     0.952756  
1  0.983756     0.942257  
2  0.984588     0.954506  
3  0.981949     0.952756  
4  0.977539     0.948381  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.174593  0.011992  0.1746 ± 0.0120  [0.1597, 0.1895]
1     Accuracy  0.941120  0.007568  0.9411 ± 0.0076  [0.9317, 0.9505]
2    Precision  0.949229  0.004750  0.9492 ± 0.0048  [0.9433, 0.9551]
3       Recall  0.932108  0.014648  0.9321 ± 0.0146  [0.9139, 0.9503]
4

# BiLSTM

In [34]:

from tensorflow.keras.layers import Bidirectional



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            LSTM(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            LSTM(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [37]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiLSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8588 - auc: 0.9205 - loss: 0.3599 - val_accuracy: 0.8696 - val_auc: 0.9363 - val_loss: 0.3196
Epoch 2/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8865 - auc: 0.9477 - loss: 0.2889 - val_accuracy: 0.8867 - val_auc: 0.9549 - val_loss: 0.2732
Epoch 3/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9064 - auc: 0.9634 - loss: 0.2421 - val_accuracy: 0.9055 - val_auc: 0.9646 - val_loss: 0.2383
Epoch 4/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9143 - auc: 0.9688 - loss: 0.2216 - val_accuracy: 0.9129 - val_auc: 0.9690 - val_loss: 0.2238
Epoch 5/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9201 - auc: 0.9716 - loss: 0.2110 - val_accuracy: 0.9103 - val_auc: 0.9685 - val_loss: 0.2247
Epoch 6/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9252 - auc: 0.9737 - loss: 0.2032 - val_accuracy: 0.9125 - val_auc: 0.9725 - val_loss: 0.2123
Epoch 7/30
572/5

In [38]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",      
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.191161  0.943570   0.955117  0.930884  0.942844  0.887425   
1     3  0.163675  0.949694   0.948517  0.951006  0.949760  0.899391   
2     7  0.168165  0.944882   0.955237  0.933508  0.944248  0.889994   
3    72  0.160712  0.942257   0.950134  0.933508  0.941748  0.884650   
4    82  0.173138  0.945319   0.958559  0.930884  0.944518  0.891010   

        AUC  Specificity  
0  0.983074     0.956255  
1  0.985066     0.948381  
2  0.985993     0.956255  
3  0.985676     0.951006  
4  0.985128     0.959755  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.171370  0.012020  0.1714 ± 0.0120  [0.1564, 0.1863]
1     Accuracy  0.945144  0.002811  0.9451 ± 0.0028  [0.9417, 0.9486]
2    Precision  0.953513  0.004104  0.9535 ± 0.0041  [0.9484, 0.9586]
3       Recall  0.935958  0.008514  0.9360 ± 0.0085  [0.9254, 0.9465]
4

# GRU

In [39]:

from tensorflow.keras.layers import GRU



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        GRU(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        GRU(
            64
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [40]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (GRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8588 - auc: 0.9205 - loss: 0.3599 - val_accuracy: 0.8696 - val_auc: 0.9363 - val_loss: 0.3196
Epoch 2/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8865 - auc: 0.9477 - loss: 0.2889 - val_accuracy: 0.8867 - val_auc: 0.9549 - val_loss: 0.2732
Epoch 3/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9064 - auc: 0.9634 - loss: 0.2421 - val_accuracy: 0.9055 - val_auc: 0.9646 - val_loss: 0.2383
Epoch 4/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9143 - auc: 0.9688 - loss: 0.2216 - val_accuracy: 0.9129 - val_auc: 0.9690 - val_loss: 0.2238
Epoch 5/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9201 - auc: 0.9716 - loss: 0.2110 - val_accuracy: 0.9103 - val_auc: 0.9685 - val_loss: 0.2247
Epoch 6/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9252 - auc: 0.9737 - loss: 0.2032 - val_accuracy: 0.9125 - val_auc: 0.9725 - val_loss: 0.2123
Epoch 7/30
572/5

In [41]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.191161  0.943570   0.955117  0.930884  0.942844  0.887425   
1     3  0.163675  0.949694   0.948517  0.951006  0.949760  0.899391   
2     7  0.168165  0.944882   0.955237  0.933508  0.944248  0.889994   
3    72  0.160712  0.942257   0.950134  0.933508  0.941748  0.884650   
4    82  0.173138  0.945319   0.958559  0.930884  0.944518  0.891010   

        AUC  Specificity  
0  0.983074     0.956255  
1  0.985066     0.948381  
2  0.985993     0.956255  
3  0.985676     0.951006  
4  0.985128     0.959755  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.171370  0.012020  0.1714 ± 0.0120  [0.1564, 0.1863]
1     Accuracy  0.945144  0.002811  0.9451 ± 0.0028  [0.9417, 0.9486]
2    Precision  0.953513  0.004104  0.9535 ± 0.0041  [0.9484, 0.9586]
3       Recall  0.935958  0.008514  0.9360 ± 0.0085  [0.9254, 0.9465]
4

# BiGRU

In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Bidirectional, Dropout, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf


def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            GRU(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            GRU(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [44]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiGRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.8687 - auc: 0.9377 - loss: 0.3175 - val_accuracy: 0.8871 - val_auc: 0.9544 - val_loss: 0.2754
Epoch 2/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9075 - auc: 0.9646 - loss: 0.2382 - val_accuracy: 0.9134 - val_auc: 0.9661 - val_loss: 0.2347
Epoch 3/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9194 - auc: 0.9728 - loss: 0.2093 - val_accuracy: 0.9291 - val_auc: 0.9756 - val_loss: 0.1956
Epoch 4/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9306 - auc: 0.9784 - loss: 0.1845 - val_accuracy: 0.9318 - val_auc: 0.9787 - val_loss: 0.1819
Epoch 5/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9345 - auc: 0.9808 - loss: 0.1734 - val_accuracy: 0.9366 - val_auc: 0.9798 - val_loss: 0.1779
Epoch 6/30
572/572 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9364 - auc: 0.9820 - loss: 0.1683 - val_accuracy: 0.9344 - val_auc: 0.9793 - val_loss: 0.1811
Epoch 7/30

In [45]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.145305  0.950569   0.956560  0.944007  0.950242  0.901215   
1     3  0.164055  0.947069   0.949033  0.944882  0.946953  0.894147   
2     7  0.152081  0.947944   0.945217  0.951006  0.948103  0.895905   
3    72  0.150170  0.950569   0.954947  0.945757  0.950330  0.901179   
4    82  0.162050  0.951444   0.964029  0.937883  0.950776  0.903219   

        AUC  Specificity  
0  0.988753     0.957130  
1  0.985542     0.949256  
2  0.987896     0.944882  
3  0.987262     0.955381  
4  0.987131     0.965004  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.154732  0.008018  0.1547 ± 0.0080  [0.1448, 0.1647]
1     Accuracy  0.949519  0.001897  0.9495 ± 0.0019  [0.9472, 0.9519]
2    Precision  0.953957  0.007243  0.9540 ± 0.0072  [0.9450, 0.9630]
3       Recall  0.944707  0.004687  0.9447 ± 0.0047  [0.9389, 0.9505]
4